# 2.1 · De una sola llamada al loop de agente
**Módulo 2 · Construye el loop básico del agente** — *Seguridad de agentes con OpenShell y NemoClaw* (Platzi × NVIDIA Deep Learning Institute)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uncrayon/curso-seguridad-agentes-nemoclaw/blob/main/clases/2.1-loop-de-agente/2.1-loop-de-agente.ipynb)

Un modelo, por sí solo, recibe texto y devuelve texto **una vez**. Un **agente** es ese modelo dentro de un **loop**: observa, decide, actúa y revisa si ya terminó.

En este notebook vas a:

1. Hacer **una sola llamada** al modelo y leer lo que devuelve (`content`, `finish_reason`, `usage`).
2. Construir un agente **sin LLM** que sale de un laberinto, y ver que el loop es solo código.
3. Cambiar **solo la función que decide** por un LLM. El loop no cambia.
4. Provocar el error típico: el modelo devuelve un movimiento inválido. El *harness* (el código que rodea al modelo) lo valida.
5. Ver que la **memoria** del modelo es solo la lista `messages` que reenvías.
6. 🚀 **Proyecto:** el primer **Soporte NemoClaw**, un chat con memoria y con límite de contexto.

*Basado en la lección 01a-loop del curso de NVIDIA DLI "Securing Agents with OpenShell and NemoClaw" (Apache-2.0).*

> **¿Sin API key?** Ejecuta todo igual: el notebook detecta que no hay key y usa un **modelo simulado (MOCK)** que imita la forma de las respuestas de la API. Con tu key `nvapi-` (clase 1.2), las mismas celdas llaman al modelo real.

> **¿No programas?** No necesitas leer el código. Presiona ▶ en cada celda, en orden, y lee el texto y lo que aparece debajo (o usa **Entorno de ejecución → Ejecutar todo** en Colab). Las celdas con ⚙️ son de preparación: se ejecutan igual, pero no necesitas abrirlas. Las palabras nuevas están en el [glosario](https://github.com/uncrayon/curso-seguridad-agentes-nemoclaw/blob/main/GLOSARIO.md).

In [ ]:
# @title ⚙️ Configuración (ejecútala siempre primero)
# --- Configuración (ejecútala siempre primero) ---------------------------------
import os, sys, json, subprocess

def _instala(paquete, modulo=None):
    try:
        __import__(modulo or paquete)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

_instala("openai")
_instala("python-dotenv", "dotenv")

# 1) Google Colab: guarda tu key en Secrets (icono de llave 🔑) con el nombre NVIDIA_API_KEY
try:
    from google.colab import userdata
    os.environ.setdefault("NVIDIA_API_KEY", userdata.get("NVIDIA_API_KEY") or "")
except Exception:
    pass
# 2) Local: archivo .env en la raíz del repo (copia .env.example) o `export NVIDIA_API_KEY=nvapi-...`
from dotenv import load_dotenv
load_dotenv(); load_dotenv("../../.env")

BASE_URL    = os.environ.get("NVIDIA_BASE_URL", "https://integrate.api.nvidia.com/v1")
MODEL       = os.environ.get("NVIDIA_MODEL", "nvidia/nemotron-3.5-lightning-30b-a3b")
EMBED_MODEL = os.environ.get("NVIDIA_EMBED_MODEL", "nvidia/nemotron-3-embed-1b")
API_KEY     = (os.environ.get("NVIDIA_API_KEY") or "").strip()
USE_MOCK    = not API_KEY

if API_KEY and "integrate.api.nvidia.com" in BASE_URL and not API_KEY.startswith("nvapi-"):
    print("⚠  Tu key no empieza con 'nvapi-'. ¿Copiaste la key completa de build.nvidia.com?")
print("Modo:", "MOCK (sin NVIDIA_API_KEY: respuestas simuladas)" if USE_MOCK else f"EN VIVO · {MODEL}")

In [ ]:
# @title ⚙️ Cliente: modelo real o simulado, con errores explicados en español
# --- Cliente: modelo real o simulado, con errores explicados en español --------
import re, time, math, hashlib
from types import SimpleNamespace as NS

def _msg(content):
    m = NS(content=content, tool_calls=None, role="assistant", reasoning_content=None)
    return NS(choices=[NS(message=m, finish_reason="stop")], usage=NS(prompt_tokens=0, completion_tokens=0, total_tokens=0))

def _call(name, args):
    tc = NS(id="call_" + hashlib.md5((name + json.dumps(args)).encode()).hexdigest()[:8], type="function",
            function=NS(name=name, arguments=json.dumps(args, ensure_ascii=False)))
    m = NS(content=None, tool_calls=[tc], role="assistant", reasoning_content=None)
    return NS(choices=[NS(message=m, finish_reason="tool_calls")], usage=NS(prompt_tokens=0, completion_tokens=0, total_tokens=0))

class MockClient:
    """Imita la FORMA de la API de chat completions. No es inteligente: sigue reglas.
    Cada notebook puede agregar reglas propias con  @regla_mock  (ver más abajo)."""
    reglas = []
    def __init__(self):
        self.chat = NS(completions=NS(create=self._create))
        self.embeddings = NS(create=self._embed)
    def _create(self, model=None, messages=None, tools=None, **kw):
        nombres = [t["function"]["name"] for t in (tools or [])]
        for r in self.reglas:
            out = r(messages, nombres)
            if out is not None:
                return out
        ultimo = messages[-1]
        texto = (ultimo.get("content") or "")
        if ultimo["role"] == "tool":
            return _msg("Según la herramienta: " + str(ultimo["content"]))
        if re.search(r"c[oó]mo me llamo|what is my name", texto, re.I):
            vistos = [re.search(r"(?:me llamo|my name is)\s+(\w+)", m.get("content") or "", re.I)
                      for m in messages[:-1] if m["role"] == "user"]
            vistos = [v.group(1) for v in vistos if v]
            return _msg(f"Te llamas {vistos[-1]}." if vistos else "No sé cómo te llamas: no me lo has dicho en esta conversación.")
        return _msg("(mock) Respuesta simulada. Configura NVIDIA_API_KEY para hablar con el modelo real.")
    def _embed(self, model=None, input=None, **kw):
        textos = [input] if isinstance(input, str) else list(input)
        return NS(data=[NS(embedding=_vector_bolsa(t), index=i) for i, t in enumerate(textos)])

def regla_mock(f):
    """Decorador: agrega una regla al modelo simulado. f(messages, nombres_de_tools) -> respuesta | None"""
    MockClient.reglas.append(f)
    return f

_VACIAS = set("que qué como cómo para por con una uno los las del este esta eso esto hago hace tengo puedo sale mis tus sus the and you what how".split())

def _vector_bolsa(texto, dim=256):
    """Embedding simulado: bolsa de palabras con hashing. Sirve para que RAG funcione offline."""
    v = [0.0] * dim
    for w in re.findall(r"\w+", texto.lower()):
        if len(w) > 2 and w not in _VACIAS:
            w = w[:6]                       # raíz burda: "reanudar" ~ "reanudé"
            v[int(hashlib.md5(w.encode()).hexdigest(), 16) % dim] += 1.0
    n = math.sqrt(sum(x * x for x in v)) or 1.0
    return [x / n for x in v]

def diagnostico(e):
    """Traduce un error de la API a: qué pasó + qué hacer. Es el 'FAQ' en código."""
    status = getattr(e, "status_code", None)
    cuerpo = str(getattr(e, "body", "") or e)[:300]
    tabla = {
        400: "400 · Solicitud inválida: algún parámetro no le gusta al modelo (¿tools en un modelo que no las soporta? ¿max_tokens muy alto?). Lee el cuerpo del error.",
        401: "401 · Key rechazada: revisa que NVIDIA_API_KEY sea la key completa (empieza con nvapi-), sin espacios ni comillas. Si la regeneraste, la vieja ya no sirve.",
        403: "403 · Tu key es válida pero no tiene acceso a ese modelo o se acabaron tus créditos. Revisa tu cuenta en build.nvidia.com o prueba otro modelo.",
        404: "404 · Ese modelo o esa ruta no existen. Revisa MODEL / EMBED_MODEL (copia el ID exacto del catálogo) y que BASE_URL termine en /v1.",
        410: "410 · El modelo fue retirado del endpoint gratuito. Cambia MODEL por uno vigente del catálogo de build.nvidia.com.",
        422: "422 · Argumentos con formato incorrecto (p. ej. falta input_type en embeddings o un esquema de tool inválido).",
        429: "429 · Demasiadas solicitudes por minuto. Espera unos segundos y reintenta; no ejecutes celdas en ráfaga.",
    }
    if status in tabla:
        return f"{tabla[status]}\n   Detalle del servidor: {cuerpo}"
    if status and 500 <= status < 600:
        return f"{status} · Falla del lado del servidor o el modelo está arrancando. Reintenta en 30-60 s; si persiste, prueba otro modelo.\n   Detalle: {cuerpo}"
    nombre = type(e).__name__
    if "Timeout" in nombre:
        return "Timeout · El modelo tardó demasiado. Reintenta, baja max_tokens o usa un modelo más rápido."
    if "Connection" in nombre:
        return "Sin conexión · No se pudo llegar a BASE_URL. Revisa tu internet, VPN/proxy corporativo o el valor de BASE_URL."
    return f"{nombre}: {cuerpo}"

if USE_MOCK:
    client = MockClient()
else:
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=60, max_retries=2)

def chat(messages, tools=None, model=None, **kw):
    """Una llamada al modelo. Devuelve la respuesta cruda (choices[0].message, finish_reason, usage)."""
    kw.setdefault("max_tokens", 4096)          # los modelos que razonan gastan tokens antes de responder
    if tools:
        kw["tools"] = tools
    try:
        r = client.chat.completions.create(model=model or MODEL, messages=messages, **kw)
    except Exception as e:
        raise RuntimeError(diagnostico(e)) from None
    ch = r.choices[0]
    if ch.finish_reason == "length" and not (ch.message.content or ch.message.tool_calls):
        print("⚠  finish_reason=length sin contenido: el modelo gastó max_tokens razonando. Sube max_tokens.")
    return r

def embed(textos, input_type="passage", model=None):
    """Embeddings. input_type='query' para preguntas, 'passage' para documentos."""
    textos = [textos] if isinstance(textos, str) else list(textos)
    try:
        r = client.embeddings.create(model=model or EMBED_MODEL, input=textos,
                                     extra_body={"input_type": input_type, "truncate": "END"}) if not USE_MOCK \
            else client.embeddings.create(model=model or EMBED_MODEL, input=textos)
    except Exception as e:
        raise RuntimeError(diagnostico(e)) from None
    return [d.embedding for d in r.data]

print("Cliente listo:", "MockClient" if USE_MOCK else BASE_URL)

> **Celda del simulador.** Solo se usa en modo MOCK: le enseña al modelo simulado a comportarse como un modelo real en *este* notebook (pedir herramientas, equivocarse a veces). Con tu key no se usa. Ejecútala y sigue; no hace falta leerla.

In [ ]:
# @title ⚙️ Simulador: reglas de ESTE notebook (solo modo MOCK)
# --- Simulador: reglas de ESTE notebook (solo modo MOCK) -------------------------
_MOVS = {"arriba": (0, -1), "abajo": (0, 1), "izquierda": (-1, 0), "derecha": (1, 0)}
_ERRORES_MOCK = []

def _bfs_mock(filas):
    from collections import deque
    ini = next((x, y) for y, f in enumerate(filas) for x, c in enumerate(f) if c == "@")
    meta = next((x, y) for y, f in enumerate(filas) for x, c in enumerate(f) if c == "G")
    cola, vistos = deque([(ini, None)]), {ini}
    while cola:
        (x, y), primero = cola.popleft()
        if (x, y) == meta:
            return primero
        for nombre, (dx, dy) in _MOVS.items():
            n = (x + dx, y + dy)
            if 0 <= n[1] < len(filas) and 0 <= n[0] < len(filas[0]) and filas[n[1]][n[0]] != "#" and n not in vistos:
                vistos.add(n); cola.append((n, primero or nombre))
    return "arriba"

@regla_mock
def _laberinto(messages, nombres):
    sistema = messages[0].get("content") or ""
    if "laberinto" not in sistema.lower():
        return None
    usuario = "\n".join(m.get("content") or "" for m in messages if m["role"] == "user")
    if "UNA palabra" not in sistema:                       # prompt flojo → respuesta conversacional
        return _msg("Te sugiero ir hacia abajo: a la derecha casi todo es pared y abajo hay un pasillo largo.")
    mapa = re.search(r"MAPA:\n(.*?)\n\n", usuario, re.S)
    if not mapa:
        return _msg("derecha")
    if "Posición: [0, 0]" in usuario and not _ERRORES_MOCK:
        _ERRORES_MOCK.append(1)
        return _msg("arriba")                               # un modelo real a veces elige una pared (solo la 1.ª vez)
    return _msg(_bfs_mock(mapa.group(1).splitlines()))

@regla_mock
def _charla(messages, nombres):
    if messages[-1]["role"] != "user":
        return None
    ultimo = messages[-1].get("content") or ""
    if re.search(r"sin estado", ultimo, re.I):
        return _msg("Significa que cada llamada es independiente: el modelo no recuerda nada y tu código debe reenviar el contexto completo en cada solicitud.")
    nombre = re.search(r"me llamo\s+(\w+)", ultimo, re.I)
    if nombre:
        return _msg(f"¡Mucho gusto, {nombre.group(1)}! Lo recordaré mientras me reenvíes esta conversación.")
    return None

_RESPUESTAS_SOPORTE = [
    (r"api key|nvapi|401|clave", "Tu key de NVIDIA Build empieza con nvapi-. Guárdala en la variable NVIDIA_API_KEY o en un archivo .env, nunca en el código. Si ves un 401, vuelve a copiarla completa, sin espacios."),
    (r"brev|instancia|apag|deten|cr[eé]dito", "Cuando no uses tu instancia de Brev, detenla: deja de cobrar cómputo, aunque el disco sigue cobrando un poco. Si ya no la necesitas, elimínala."),
    (r"nemoclaw|openshell|openclaw", "OpenClaw es el agente, OpenShell es el sandbox que lo contiene y NemoClaw es el blueprint que instala y conecta todo."),
]

@regla_mock
def _soporte(messages, nombres):
    if "Soporte NemoClaw" not in (messages[0].get("content") or "") or messages[-1]["role"] != "user":
        return None
    ultimo = messages[-1].get("content") or ""
    if re.search(r"c[oó]mo me llamo", ultimo, re.I):
        return None                                         # lo resuelve la regla de memoria del simulador
    for patron, respuesta in _RESPUESTAS_SOPORTE:
        if re.search(patron, ultimo, re.I):
            return _msg(respuesta)
    return _msg("Todavía no tengo ese dato. Revisa FAQ.md del curso o pregunta en los comentarios de la clase.")

print("Reglas del simulador listas:", len(MockClient.reglas))

## 1 · Una sola llamada: el modelo no tiene estado

Así se ve **una** llamada. Fíjate en tres campos de la respuesta:

- `content`: el texto de la respuesta.
- `finish_reason`: por qué se detuvo el modelo (`stop` = terminó; `length` = se quedó sin tokens; `tool_calls` = pide una herramienta, lo vemos en la clase 2.2).
- `usage`: cuántos tokens gastaste (en MOCK sale en 0).

Algunos modelos, como Nemotron, también devuelven `reasoning_content`: el razonamiento previo a la respuesta.

In [ ]:
pregunta = [
    {"role": "system", "content": "Eres un asistente conciso. Responde en una sola oración."},
    {"role": "user", "content": "¿Qué significa que una API de LLM sea 'sin estado'?"},
]
r = chat(pregunta)
c = r.choices[0]
print("finish_reason:", c.finish_reason)
print("razonamiento :", (getattr(c.message, "reasoning_content", None) or "(no vino)")[:200])
print("respuesta    :", c.message.content)
print("usage        :", r.usage)

**✏️ Tu turno.** Cambia la pregunta por `"¿Qué te pregunté hace un momento?"` y vuelve a ejecutar. El modelo no lo sabe: **cada llamada empieza de cero**. Guarda esta idea; vuelve en la sección 5.

## 2 · El entorno: un laberinto

Un agente vive en un **entorno**. El entorno guarda el estado real; el agente solo recibe **observaciones** y manda **acciones**.

Leyenda: `S` inicio · `G` meta · `#` pared · `@` el agente · `·` celdas visitadas.

In [ ]:
LABERINTO = [
    "S.#.....",
    ".##.###.",
    "....#...",
    ".####.#.",
    "......#G",
]
MOVIMIENTOS = {"arriba": (0, -1), "abajo": (0, 1), "izquierda": (-1, 0), "derecha": (1, 0)}

class Laberinto:
    """El ENTORNO. Guarda el estado; el agente solo ve observaciones y manda acciones."""
    def __init__(self, mapa=LABERINTO):
        self.mapa = mapa
        self.alto, self.ancho = len(mapa), len(mapa[0])
        for y, fila in enumerate(mapa):
            for x, ch in enumerate(fila):
                if ch == "S": self.pos = (x, y)
                if ch == "G": self.meta = (x, y)
        self.visitadas = [self.pos]

    def legales(self):
        """Movimientos que no chocan con una pared ni salen del mapa."""
        out = []
        for nombre, (dx, dy) in MOVIMIENTOS.items():
            x, y = self.pos[0] + dx, self.pos[1] + dy
            if 0 <= x < self.ancho and 0 <= y < self.alto and self.mapa[y][x] != "#":
                out.append(nombre)
        return out

    def dibujar(self):
        filas = []
        for y, fila in enumerate(self.mapa):
            filas.append("".join("@" if (x, y) == self.pos else
                                 "·" if (x, y) in self.visitadas and fila[x] in ".S" else fila[x]
                                 for x in range(self.ancho)))
        return "\n".join(filas)

    def observar(self):
        """Todo lo que el agente tiene permitido saber."""
        return {"pos": list(self.pos), "meta": list(self.meta), "legales": self.legales(),
                "mapa": self.dibujar()}

    def actuar(self, accion):
        """La única forma de cambiar el mundo. El entorno se protege: rechaza movimientos inválidos."""
        if accion not in self.legales():
            raise ValueError(f"Movimiento inválido: {accion!r}. Desde {self.pos} solo puedes: {self.legales()}")
        dx, dy = MOVIMIENTOS[accion]
        self.pos = (self.pos[0] + dx, self.pos[1] + dy)
        self.visitadas.append(self.pos)
        return self.pos == self.meta          # ¿llegó?

env = Laberinto()
print(env.dibujar())
print({k: v for k, v in env.observar().items() if k != "mapa"})

Deberías ver el mapa con `@` en la esquina superior izquierda y `legales: ['abajo', 'derecha']`.

## 3 · Un agente sin LLM

El loop tiene cuatro partes: **observar → decidir → actuar → ¿parar?** Fíjate que `correr_agente` no sabe nada de laberintos ni de modelos: recibe la función `decidir` como parámetro.

In [ ]:
def correr_agente(env, decidir, max_pasos=30, verbose=True):
    """EL LOOP. No cambia nunca, decida quien decida."""
    for paso in range(1, max_pasos + 1):
        obs = env.observar()                       # 1. observar
        accion = decidir(obs)                      # 2. decidir   (lo único que vamos a cambiar)
        if accion not in obs["legales"]:           # el harness valida ANTES de actuar
            if verbose: print(f"paso {paso:2d}: {accion!r} no es válido → no se aplica")
            continue
        llego = env.actuar(accion)                 # 3. actuar
        if verbose: print(f"paso {paso:2d}: {accion:9s} → {env.pos}")
        if llego:                                  # 4. ¿parar?
            print(f"✅ Meta en {paso} pasos.")
            return paso
    print(f"⛔ Se acabó el presupuesto de {max_pasos} pasos sin llegar.")
    return None

# Decisión 1: una regla fija. Funciona en un pasillo recto... y aquí se atora.
def siempre_derecha(obs):
    return "derecha"

correr_agente(Laberinto(), siempre_derecha, max_pasos=6)

La regla fija se atora contra la primera pared. Por eso el loop **necesita un tope de pasos**: sin él, este agente se quedaría intentando para siempre.

Ahora cambiamos **solo la decisión** por una búsqueda en anchura (BFS), que calcula el camino más corto leyendo el mapa de la observación.

In [ ]:
from collections import deque

def primer_paso_bfs(filas):
    """Primer movimiento del camino más corto de @ a G, leyendo el mapa como texto."""
    ini = next((x, y) for y, f in enumerate(filas) for x, ch in enumerate(f) if ch == "@")
    meta = next((x, y) for y, f in enumerate(filas) for x, ch in enumerate(f) if ch == "G")
    cola, vistos = deque([(ini, None)]), {ini}
    while cola:
        (x, y), primero = cola.popleft()
        if (x, y) == meta:
            return primero
        for nombre, (dx, dy) in MOVIMIENTOS.items():
            n = (x + dx, y + dy)
            if 0 <= n[1] < len(filas) and 0 <= n[0] < len(filas[0]) and filas[n[1]][n[0]] != "#" and n not in vistos:
                vistos.add(n); cola.append((n, primero or nombre))

def decidir_bfs(obs):
    return primer_paso_bfs(obs["mapa"].splitlines())

env = Laberinto()
pasos_bfs = correr_agente(env, decidir_bfs)
print(env.dibujar())

## 4 · El mismo loop, pero decide un LLM

Ahora la decisión la toma el modelo. Le describimos la observación en texto y le pedimos **una palabra**. El loop, el entorno y la condición de parada **no cambian**.

Tres detalles del harness que separan un juguete de algo confiable:

1. `extraer_movimiento` busca un movimiento dentro del texto, aunque el modelo agregue un punto o una frase.
2. Si el movimiento no es legal, **se lo decimos al modelo** y le damos un reintento.
3. `SIN_RAZONAR` apaga el razonamiento largo de Nemotron para esta tarea simple (es el perfil `direct` del curso de NVIDIA).

> En MOCK el "modelo" usa BFS por dentro y se equivoca a propósito en el primer paso. Con un modelo real vas a ver titubeos distintos en cada ejecución.

In [ ]:
SISTEMA_LABERINTO = """Controlas a un agente dentro de un laberinto.
Leyenda: # pared · . pasillo · @ tú · · celda visitada · G meta. x crece a la derecha, y crece hacia abajo.
Responde con UNA palabra: uno de los movimientos legales. Sin explicación.
Estrategia: prefiere celdas no visitadas y acércate a G."""

SIN_RAZONAR = {"chat_template_kwargs": {"enable_thinking": False}}   # perfil "direct" de Nemotron

def describir(obs):
    return (f"MAPA:\n{obs['mapa']}\n\n"
            f"Posición: {obs['pos']} · Meta: {obs['meta']}\n"
            f"Movimientos legales: {', '.join(obs['legales'])}")

def extraer_movimiento(texto):
    """Primera palabra de movimiento dentro de la respuesta. Si no hay ninguna, devuelve el texto tal cual."""
    for palabra in re.findall(r"[a-záéíóú]+", (texto or "").lower()):
        if palabra in MOVIMIENTOS:
            return palabra
    return (texto or "").strip()

def decidir_llm(obs, reintentos=1):
    mensajes = [{"role": "system", "content": SISTEMA_LABERINTO},
                {"role": "user", "content": describir(obs)}]
    for intento in range(reintentos + 1):
        r = chat(mensajes, temperature=0, max_tokens=256, extra_body=SIN_RAZONAR)
        texto = r.choices[0].message.content or ""
        mov = extraer_movimiento(texto)
        if mov in obs["legales"]:
            return mov
        print(f"   ⚠ el modelo respondió {texto[:60]!r}: no es legal. Se lo decimos y reintenta.")
        mensajes += [{"role": "assistant", "content": texto},
                     {"role": "user", "content": f"{mov!r} no es un movimiento válido. Elige exactamente una de: {', '.join(obs['legales'])}"}]
    return mov          # si sigue mal, el loop lo rechaza y no mueve al agente

env = Laberinto()
pasos_llm = correr_agente(env, decidir_llm, max_pasos=30)
print(env.dibujar())
print(f"\nBFS: {pasos_bfs} pasos · LLM: {pasos_llm} pasos")

**¿Por qué usar un LLM si BFS es perfecto y gratis?** Porque BFS solo sirve para laberintos. Un LLM puede decidir en situaciones para las que nadie escribió un algoritmo. Ese es el intercambio: flexibilidad a cambio de errores que tu harness debe atrapar.

**✏️ Tu turno** (son los mismos experimentos de la sección *Antes de continuar* de la lección 01a de NVIDIA):

1. Quita la línea `Estrategia: ...` de `SISTEMA_LABERINTO` y vuelve a correr. ¿Cuántos pasos tarda ahora?
2. En `describir`, quita el `MAPA` y deja solo posición, meta y movimientos legales. Es como `includeHistory:false` en el curso de NVIDIA: el modelo pierde contexto. ¿Llega a la meta?
3. Cambia `temperature=0` por `temperature=1.0`. ¿Es igual en cada ejecución?

In [ ]:
# ✏️ Edita aquí tu versión del prompt y compárala
SISTEMA_LABERINTO_MIO = SISTEMA_LABERINTO.replace("Estrategia: prefiere celdas no visitadas y acércate a G.", "")

def decidir_llm_mio(obs):
    r = chat([{"role": "system", "content": SISTEMA_LABERINTO_MIO},
              {"role": "user", "content": describir(obs)}], temperature=0, max_tokens=256, extra_body=SIN_RAZONAR)
    return extraer_movimiento(r.choices[0].message.content)

pasos_mio = correr_agente(Laberinto(), decidir_llm_mio, max_pasos=30, verbose=False)
print(f"Prompt original: {pasos_llm} pasos · tu prompt: {pasos_mio} pasos")

## 5 · 🧯 Cuando algo falla: el modelo devuelve un movimiento inválido

Vamos a provocar el error más común de un agente principiante: **confiar en el texto del modelo**. Usamos un prompt flojo (no pide formato) y pasamos la respuesta directo al entorno, sin validar.

In [ ]:
SISTEMA_FLOJO = "Ayuda a un agente a salir de un laberinto."

def decidir_ingenuo(obs):
    r = chat([{"role": "system", "content": SISTEMA_FLOJO},
              {"role": "user", "content": describir(obs)}], max_tokens=256, extra_body=SIN_RAZONAR)
    return (r.choices[0].message.content or "").strip()     # confiamos ciegamente en el texto 😬

env = Laberinto()
accion = decidir_ingenuo(env.observar())
print("El modelo dijo:", repr(accion[:150]))
try:
    env.actuar(accion)                     # sin validar: directo al entorno
except ValueError as e:
    print("\n💥", str(e)[:200])

Deberías ver `💥 Movimiento inválido: 'Te sugiero ir hacia abajo...'`. El modelo respondió algo razonable para un humano, pero tu código esperaba una palabra exacta. (Si en vivo tu modelo respondió una sola palabra legal, no hubo error: vuelve a ejecutar o cambia el prompt; el riesgo sigue ahí).

**La solución no es "un mejor prompt", es un harness que valida:**

In [ ]:
mov = extraer_movimiento(accion)
legales = env.observar()["legales"]
print(f"Movimiento extraído: {mov!r} · legales: {legales}")
if mov in legales:
    env.actuar(mov)
    print("✅ Movimiento aplicado. Posición:", env.pos)
else:
    print("⛔ Sigue siendo inválido: no se aplica y se le pide al modelo que corrija (como en decidir_llm).")

Regla de oro del curso: **la salida del modelo es una entrada no confiable**. El esquema y el prompt piden un formato; el harness es el que lo hace cumplir. En el Módulo 5 verás la misma idea aplicada a la seguridad.

## 6 · La memoria es `messages`

El modelo no recuerda nada entre llamadas. Lo que llamamos "memoria" es la lista `messages` que tu código guarda y **reenvía completa** en cada turno.

In [ ]:
historial = [{"role": "system", "content": "Eres conciso. Responde en una oración."}]
for q in ["Me llamo Ada. Recuérdalo.", "¿Cómo me llamo?"]:
    historial.append({"role": "user", "content": q})
    r = chat(historial)
    historial.append({"role": "assistant", "content": r.choices[0].message.content})
    print("👤", q)
    print("🤖", r.choices[0].message.content, "\n")

print("--- la misma pregunta, SIN historial ---")
r = chat([{"role": "system", "content": "Eres conciso."}, {"role": "user", "content": "¿Cómo me llamo?"}])
print("🤖", r.choices[0].message.content)

print("\n--- lo que realmente enviamos en el turno 2 ---")
print(json.dumps(historial[:4], ensure_ascii=False, indent=2))

Consecuencia práctica: **cada turno cuesta más que el anterior**, porque reenvías todo. Por eso cualquier chat real necesita un límite de contexto. Eso es lo que construyes ahora.

## 7 · 🚀 Avance del proyecto: Soporte NemoClaw con memoria

El proyecto del curso es **Soporte NemoClaw**, un agente que responde dudas de estudiantes sobre NemoClaw, Brev y el curso. Hoy es un chat con:

- un **system prompt** que define su rol y sus reglas,
- **memoria** (el historial),
- una **ventana**: solo reenvía los últimos `max_mensajes` mensajes, y
- un **límite de turnos** por sesión, para no gastar créditos sin control.

In [ ]:
SISTEMA_SOPORTE = """Eres "Soporte NemoClaw", el asistente del curso Seguridad de agentes con OpenShell y NemoClaw (Platzi × NVIDIA).
Respondes dudas sobre NemoClaw, OpenClaw, OpenShell, Brev y las API keys de NVIDIA Build.
Reglas: responde en español y en máximo 3 oraciones. Si no sabes algo, dilo y sugiere revisar FAQ.md.
Nunca pidas ni repitas una API key completa."""

class ChatSoporte:
    def __init__(self, max_mensajes=8, max_turnos=20):
        self.historial = [{"role": "system", "content": SISTEMA_SOPORTE}]
        self.max_mensajes = max_mensajes      # ventana: cuántos mensajes (sin contar el system) se reenvían
        self.max_turnos = max_turnos          # tope de preguntas por sesión
        self.turnos = 0

    def _recortar(self):
        sistema, resto = self.historial[0], self.historial[1:]
        resto = resto[-self.max_mensajes:]
        while resto and resto[0]["role"] != "user":   # la ventana siempre empieza con un mensaje del usuario
            resto = resto[1:]
        self.historial = [sistema] + resto

    def preguntar(self, texto):
        if self.turnos >= self.max_turnos:
            return "⛔ Llegaste al límite de esta sesión. Abre una nueva con ChatSoporte()."
        self.turnos += 1
        self.historial.append({"role": "user", "content": texto})
        self._recortar()
        r = chat(self.historial)
        respuesta = r.choices[0].message.content or f"(sin contenido; finish_reason={r.choices[0].finish_reason})"
        self.historial.append({"role": "assistant", "content": respuesta})
        return respuesta

CONVERSACION = [
    "Hola, me llamo Ada.",
    "¿Dónde guardo mi API key?",
    "¿Qué hago con mi instancia de Brev cuando no la uso?",
    "¿Cómo me llamo?",
]
for ventana in (4, 20):
    print(f"========== ventana de {ventana} mensajes ==========")
    soporte = ChatSoporte(max_mensajes=ventana)
    for q in CONVERSACION:
        print("👤", q)
        print("🤖", soporte.preguntar(q))
    print(f"(historial guardado: {len(soporte.historial) - 1} mensajes + el system)\n")

Con una ventana de 4 mensajes, Soporte NemoClaw **olvida tu nombre**: ese mensaje ya no se reenvió. Con 20 lo recuerda, pero cada llamada es más larga y más cara. Elegir la ventana es elegir entre memoria y costo.

**✏️ Conversa tú.** Cambia `INTERACTIVO = True` y ejecuta. Escribe `salir` para terminar.

In [ ]:
INTERACTIVO = False
if INTERACTIVO:
    soporte = ChatSoporte(max_mensajes=8, max_turnos=10)
    while True:
        q = input("👤 ")
        if q.strip().lower() in {"salir", "exit", "q"}:
            break
        print("🤖", soporte.preguntar(q))
else:
    print("Modo interactivo apagado. Pon INTERACTIVO = True para conversar.")

## 🏆 Reto: memoria de hechos

Con `max_mensajes=4`, Soporte NemoClaw olvida tu nombre. Haz que **no lo olvide** sin agrandar la ventana.

Pista: cuando el usuario diga "me llamo X", guarda `X` en un diccionario `hechos` y agrégalo al system prompt en cada llamada (por ejemplo: `"Datos del usuario: nombre=Ada"`).

**Criterio de éxito:** con `max_mensajes=4` y la `CONVERSACION` de arriba más dos preguntas extra, el bot responde bien "¿Cómo me llamo?" y `len(soporte.historial)` nunca pasa de 5 (4 + el system). Pruébalo en vivo: el simulador no lee el system prompt.

In [ ]:
# ✏️ Tu solución aquí
class ChatSoporteConHechos(ChatSoporte):
    pass

## Resumen

- Un agente es un loop: **observar → decidir → actuar → ¿parar?**, con un tope de pasos.
- La función de decisión puede ser una regla, una búsqueda o un LLM. **El loop no cambia.**
- La salida del modelo es texto no confiable: **el harness valida** antes de actuar.
- El modelo no tiene estado: la memoria es la lista `messages` que reenvías, y cuesta tokens.

**Siguiente clase (2.2):** el modelo deja de elegir solo movimientos y empieza a **pedir herramientas**. Vas a construir el loop ReAct desde cero.